# Retail Sales & Inventory — Data Cleaning

The goal of this notebook is to clean the raw data before checking its quality or using it.

We will :
- fix data types
- handle missing values
- handle duplicate records and IDs
- Standardise fields
- validate records
- Create derived fields
- Export cleaned data to `data/cleaned/`

In [18]:
import pandas as pd

# Path to the raw data
base = "C:/Users/selin/anaconda_projects/data science/Retail-Sales-Inventory-Project/data/raw"

sales = pd.read_csv(f"{base}/sales.csv")
products = pd.read_csv(f"{base}/products.csv")
stores = pd.read_csv(f"{base}/stores.csv")
inventory = pd.read_csv(f"{base}/inventory.csv")

In [19]:
# Work on copies so the raw dataframes stay untouched
sales_clean = sales.copy()
products_clean = products.copy()
stores_clean = stores.copy()
inventory_clean = inventory.copy()


In [20]:
# Check that price and cost columns are numeric

print("Product Price data type:", products_clean["Product_Price"].dtype)
print("Product Cost data type:", products_clean["Product_Cost"].dtype)

print("\nFirst 5 Product Prices:")
print(products_clean["Product_Price"].head())

print("\nFirst 5 Product Costs:")
print(products_clean["Product_Cost"].head())

Product Price data type: object
Product Cost data type: object

First 5 Product Prices:
0    $15.99 
1    $12.99 
2     $3.99 
3    $12.99 
4     $9.99 
Name: Product_Price, dtype: object

First 5 Product Costs:
0    $9.99 
1    $9.99 
2    $1.99 
3    $9.99 
4    $7.99 
Name: Product_Cost, dtype: object


In [21]:
# Strip "$" and "," then convert to float
products_clean["Product_Price"] = pd.to_numeric(
    products_clean["Product_Price"].str.replace("$", "", regex=False).str.replace(",", "", regex=False),
    errors="coerce"
)

products_clean["Product_Cost"] = pd.to_numeric(
    products_clean["Product_Cost"].str.replace("$", "", regex=False).str.replace(",", "", regex=False),
    errors="coerce"
)

# Confirm conversion worked
print(products_clean["Product_Price"].dtype)
print(products_clean["Product_Cost"].dtype)

print(products_clean[["Product_Price", "Product_Cost"]].head())

float64
float64
   Product_Price  Product_Cost
0          15.99          9.99
1          12.99          9.99
2           3.99          1.99
3          12.99          9.99
4           9.99          7.99


In [22]:
# bad values become NaT instead of crashing
sales_clean["Date"] = pd.to_datetime(sales_clean["Date"], errors="coerce")
stores_clean["Store_Open_Date"] = pd.to_datetime(stores_clean["Store_Open_Date"], errors="coerce")
print("Invalid dates:",
      sales_clean["Date"].isna().sum())

print("Earliest date:",
      sales_clean["Date"].min())

print("Latest date:",
      sales_clean["Date"].max())

# Confirm conversion worked
print(sales_clean["Date"].dtype) 
print(stores_clean["Store_Open_Date"].dtype)

Invalid dates: 0
Earliest date: 2022-01-01 00:00:00
Latest date: 2023-09-30 00:00:00
datetime64[ns]
datetime64[ns]


In [23]:
# Confirm no nulls remain after cleaning
print("Sales missing values:")
print(sales_clean.isnull().sum())

print("\nProducts missing values:")
print(products_clean.isnull().sum())

print("\nStores missing values:")
print(stores_clean.isnull().sum())

print("\nInventory missing values:")
print(inventory_clean.isnull().sum())

Sales missing values:
Sale_ID       0
Date          0
Store_ID      0
Product_ID    0
Units         0
dtype: int64

Products missing values:
Product_ID          0
Product_Name        0
Product_Category    0
Product_Cost        0
Product_Price       0
dtype: int64

Stores missing values:
Store_ID           0
Store_Name         0
Store_City         0
Store_Location     0
Store_Open_Date    0
dtype: int64

Inventory missing values:
Store_ID         0
Product_ID       0
Stock_On_Hand    0
dtype: int64


In [24]:
# Fully duplicated rows (all columns identical)
print("Sales duplicates:", sales_clean.duplicated().sum())
print("Products duplicates:", products_clean.duplicated().sum())
print("Stores duplicates:", stores_clean.duplicated().sum())
print("Inventory duplicates:", inventory_clean.duplicated().sum())

print("Duplicate Sale_IDs:", sales_clean["Sale_ID"].duplicated().sum())

Sales duplicates: 0
Products duplicates: 0
Stores duplicates: 0
Inventory duplicates: 0
Duplicate Sale_IDs: 0


In [25]:
# Negatives = impossible values (would indicate data-entry errors)
print("Negative units:", (sales_clean["Units"] < 0).sum())
print("Negative prices:", (products_clean["Product_Price"] < 0).sum())
print("Negative costs:", (products_clean["Product_Cost"] < 0).sum())
print("Negative stock:", (inventory_clean["Stock_On_Hand"] < 0).sum())

# Zeros: meaningful for stock (out-of-stock) but suspicious for prices/costs/units
print("Zero units:", (sales_clean["Units"] == 0).sum())
print("Zero prices:", (products_clean["Product_Price"] == 0).sum())
print("Zero costs:", (products_clean["Product_Cost"] == 0).sum())
print("Zero stock:", (inventory_clean["Stock_On_Hand"] == 0).sum())

Negative units: 0
Negative prices: 0
Negative costs: 0
Negative stock: 0
Zero units: 0
Zero prices: 0
Zero costs: 0
Zero stock: 77


In [26]:
print(inventory_clean[inventory_clean["Stock_On_Hand"] == 0].head(10))

     Store_ID  Product_ID  Stock_On_Hand
1           1           2              0
4           1           5              0
33          1          34              0
35          2           1              0
41          2           9              0
43          2          11              0
76          3          15              0
87          3          29              0
126         4          34              0
136         5          11              0


In [27]:
# Pairs of (dataframe, columns_to_clean)
for df, cols in [
    (products_clean, ["Product_Name", "Product_Category"]),
    (stores_clean,   ["Store_Name", "Store_City", "Store_Location"]),
]:# remove leading/trailing spaces only
    for c in cols:
        df[c] = df[c].str.strip()

# 1. Confirm dtypes stayed as string (object) — no accidental conversion
print(products_clean[["Product_Name", "Product_Category"]].dtypes)
print(stores_clean[["Store_Name", "Store_City", "Store_Location"]].dtypes)

# 2. Eyeball the actual category values
#    If standardisation worked, we should see clean, unique labels
print(products_clean["Product_Category"].value_counts(dropna=False))
print(stores_clean["Store_Location"].value_counts(dropna=False))

Product_Name        object
Product_Category    object
dtype: object
Store_Name        object
Store_City        object
Store_Location    object
dtype: object
Product_Category
Toys                 9
Art & Crafts         8
Games                8
Sports & Outdoors    7
Electronics          3
Name: count, dtype: int64
Store_Location
Downtown       29
Commercial     12
Residential     6
Airport         3
Name: count, dtype: int64


In [29]:
from pathlib import Path

# Output folder — creates it if it doesn't exist
output_dir = Path(
    "C:/Users/selin/anaconda_projects/data science/Retail-Sales-Inventory-Project/data/cleaned"
)

output_dir.mkdir(parents=True, exist_ok=True)

# Save each cleaned table (index=False to skip pandas' row numbers)
sales_clean.to_csv(output_dir / "sales_clean.csv", index=False)
products_clean.to_csv(output_dir / "products_clean.csv", index=False)
stores_clean.to_csv(output_dir / "stores_clean.csv", index=False)
inventory_clean.to_csv(output_dir / "inventory_clean.csv", index=False)

print("Cleaned files saved successfully.")

Cleaned files saved successfully.


## Cleaning Summary

| Table | Change | Rows |
|-------|--------|------|
| products | Stripped `$`/`,` from Product_Price, Product_Cost → float | 35 |
| sales | Parsed `Date` → datetime | 829,262 |
| stores | Parsed `Store_Open_Date` → datetime | 50 |
| inventory | (no changes) | 1,593 |

**Data quality verdict:** clean — 0 missing, 0 duplicates, 0 negatives.

**Open items:** 77 rows with zero stock — assumed valid out-of-stock state.